

# Part 2: 2D Wavefront Propagation & Illumination

## **2.1 Time-Domain Wave Propagation (FDTD)**

**User Observation:**
> "I want to include a forward model for wavefront propagation simulation... It solves Wave Operator $L=[D_{xx} + D_{yy}] + \omega^2/C^2(x,y)$."

**Theory: The 2D Acoustic Wave Equation**
We simulate the propagation of the WiFi signal using the scalar wave equation solved via the Finite-Difference Time-Domain (FDTD) method.

**The Equation:**
$$
\frac{\partial^2 u}{\partial t^2} = c(x,z)^2 \nabla^2 u
$$

**Numerical Discretization (Stencil):**
We use a standard second-order central difference approximation:

$$
u_{i,j}^{n+1} = 2u_{i,j}^n - u_{i,j}^{n-1} + \left( \frac{c \Delta t}{\Delta x} \right)^2 \left[ u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n - 4u_{i,j}^n \right]
$$

**User Observation (Stability):**
> "There is dispersion in the mesh, it seems the Courant condition was violated... Since you used FDTD solver... there is no way to add regularization to the solver to avoid instability right?"

**Correction (Regularization):**
We *can* add regularization. We implemented **Artificial Viscosity** (smoothing) in the update step to suppress high-frequency checkerboard noise. This acts as a low-pass filter in space:

$$
u_{i,j}^{n+1} \leftarrow (1-\eta)u_{i,j}^{n+1} + \eta (\text{Average of Neighbors})
$$

**User Observation (Wrap-around):**
> "There is wraparound in time for the wavefront it appears reflecting before being propagated. So add padding and then discard padding after."

**Correction (Padded Domain):**
We implemented a **Padded Solver**. A hidden "Sponge Layer" (padding) is added around the visible room. Waves entering this zone are damped ($ \times 0.9$) and die before reaching the periodic boundary.

---

## **2.2 Illumination Pattern (Steady-State Coverage)**

**User Observation:**
> "You forgot to add the amplitude envelope illumination code... This can be done approximating the problem as a diffusion equation problem."

**Theory: Signal Amplitude as Diffusion**
To visualize the steady-state signal strength (coverage map) without solving the computationally expensive Helmholtz equation, we approximate the signal envelope $A(x,z)$ using the **Steady-State Diffusion Equation** with decay. This treats the antenna as a heat source and the concrete column as an insulator.

**The Equation (Modified Fick's Law):**
$$
-D(x,z) \nabla^2 A(x,z) + \gamma A(x,z) = S(x,z)
$$

* $A(x,z)$: Signal Amplitude.
* $D(x,z)$: Diffusivity (High in air, Low in concrete).
* $\gamma$: Path loss / Decay factor.
* $S(x,z)$: Source term.

**User Observation (Stiffness):**
> "The constant for diffusion you defined seems to make the system stiff... Can you use a similar solver then as in the FDTD propagation problem?"

**Correction (Direct Solver):**
Time-stepping diffusion problems to steady-state is computationally inefficient and prone to stiffness (static results). We replaced the iterative ADI solver with a **Direct Sparse Solver** (`scipy.sparse.linalg.spsolve`). This solves the linear system $\mathbf{M}\mathbf{x}=\mathbf{b}$ in one shot, guaranteeing a stable solution without artifacts or convergence issues.

**References (Part 2):**
1.  **Taflove, A. & Hagness, S. C.** (2005). *Computational Electrodynamics: The Finite-Difference Time-Domain Method*. Artech House.
2.  **Durgin, G.** (2003). *Space-Time Wireless Channels*. Prentice Hall, Chapter 4: "Diffuse Scattering".
3.  **Peaceman, D. W., & Rachford, H. H.** (1955). "The numerical solution of parabolic and elliptic differential equations." *J. Soc. Ind. Appl. Math*.

# Part 2: 2D Wavefront Propagation & Signal Heatmap

## **2.1 Time-Domain Wave Propagation (FDTD)**

**User Observation:**
> "I want to include a forward model for wavefront propagation simulation... It solves Wave Operator $L=[D_{xx} + D_{yy}] + \omega^2/C^2(x,y)$."

**Theory: The 2D Acoustic Wave Equation**
We simulate the propagation of the WiFi signal using the scalar wave equation solved via the Finite-Difference Time-Domain (FDTD) method.

**The Equation:**
$$
\frac{\partial^2 u}{\partial t^2} = c(x,z)^2 \nabla^2 u
$$

**Numerical Discretization (Stencil):**
We use a standard second-order central difference approximation:
$$
u_{i,j}^{n+1} = 2u_{i,j}^n - u_{i,j}^{n-1} + \left( \frac{c \Delta t}{\Delta x} \right)^2 \left[ u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n - 4u_{i,j}^n \right]
$$

**User Observation (Stability):**
> "There is dispersion in the mesh, it seems the Courant condition was violated... Since you used FDTD solver... there is no way to add regularization to the solver to avoid instability right?"

**Correction (Regularization):**
We *can* add regularization. We implemented **Artificial Viscosity** (smoothing) in the update step to suppress high-frequency checkerboard noise:

$$
u_{i,j}^{n+1} \leftarrow (1-\eta)u_{i,j}^{n+1} + \eta (\text{Average of Neighbors})
$$

**User Observation (Wrap-around):**
> "There is wraparound in time for the wavefront it appears reflecting before being propagated. So add padding and then discard padding after."

**Correction (Padded Domain):**
We implemented a **Padded Solver**. A hidden "Sponge Layer" of 90 pixels is added around the visible room. Waves entering this zone are damped ($ \times 0.9$) and die before reaching the periodic boundary.

---

## **2.2 Illumination Pattern (Steady-State Diffusion)**

**User Observation:**
> "You forgot to add the amplitude envelope illumination code... This can be done approximating the problem as a diffusion equation problem."

**Theory: Signal Amplitude as Diffusion**
To visualize the steady-state signal strength (coverage map), we approximate the signal envelope $A(x,z)$ using the **Steady-State Diffusion Equation** with decay. This treats the antenna as a heat source and the concrete column as an insulator.

**The Equation (Modified Fick's Law):**
$$
-D(x,z) \nabla^2 A(x,z) + \gamma A(x,z) = S(x,z)
$$

* $A(x,z)$: Signal Amplitude.
* $D(x,z)$: Diffusivity (High in air, Low in concrete).
* $\gamma$: Path loss / Decay factor.
* $S(x,z)$: Source term.

**User Observation (Stiffness):**
> "The constant for diffusion you defined seems to make the system stiff... Can you use a similar solver then as in the FDTD propagation problem?"

**Correction (Direct Solver):**
Time-stepping diffusion problems to steady-state is computationally inefficient and prone to stiffness. We replaced the iterative solver with a **Direct Sparse Solver** (`scipy.sparse.linalg.spsolve`). This solves the linear system $\mathbf{M}\mathbf{x}=\mathbf{b}$ in one shot, guaranteeing a stable solution without artifacts.

---

**References:**
1.  **Taflove, A. & Hagness, S. C.** (2005). *Computational Electrodynamics: The Finite-Difference Time-Domain Method*. Artech House.
2.  **Durgin, G.** (2003). *Space-Time Wireless Channels*. Prentice Hall, Chapter 4: "Diffuse Scattering".
3.  **Peaceman, D. W., & Rachford, H. H.** (1955). "The numerical solution of parabolic and elliptic differential equations." *J. Soc. Ind. Appl. Math*.

In [ ]:
# 1. Install FFmpeg (Required to save MP4 video)
!apt-get install ffmpeg -y

# 2. Run the Simulation
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
import shutil

# --- PASTE THE REST OF THE "PADDED SOLVER" SCRIPT HERE ---
# (Copy the script from my previous response)

In [ ]:
# @title B) Run FDTD Simulation (Generates Video)
# 1. Install System Dependencies
import os
import subprocess

# Check if ffmpeg is installed, if not, install it.
if subprocess.call(['which', 'ffmpeg']) != 0:
    print("Installing ffmpeg...")
    os.system('apt-get update -qq')
    os.system('apt-get install -y ffmpeg')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
from IPython.display import Video

# --- CONFIGURATION (High-Res & Padded) ---
# Room Dimensions
L_X, L_Z = 10.0, 10.0
NX_CORE, NZ_CORE = 600, 600  # High Resolution Core
DX, DZ = L_X / NX_CORE, L_Z / NZ_CORE

# PADDING (Sponge Layer to stop wrap-around artifacts)
PAD = 90
NX_TOT = NX_CORE + 2 * PAD
NZ_TOT = NZ_CORE + 2 * PAD

# Physics
C_AIR = 343.0
# Stability: Reduced DT for the 600x600 mesh
DT = 0.00003  # 30 microseconds
STEPS = 1500  # Longer duration for finer steps
F_PEAK = 80.0 # Source Frequency

print(f"Simulation Init: {NX_TOT}x{NZ_TOT} Grid. Stability check: DT={DT}")

class PaddedFDTDSolver:
    def __init__(self, nx_tot, nz_tot, dx, dz, pad):
        self.nx = nx_tot
        self.nz = nz_tot
        self.dx = dx
        self.dz = dz
        self.pad = pad

        # Arrays (Using Float32 for memory efficiency)
        self.u_prev = np.zeros((nx_tot, nz_tot), dtype=np.float32)
        self.u      = np.zeros((nx_tot, nz_tot), dtype=np.float32)
        self.u_next = np.zeros((nx_tot, nz_tot), dtype=np.float32)

        # Velocity Model
        self.vel = np.ones((nx_tot, nz_tot), dtype=np.float32) * C_AIR
        self._build_geometry()

        self.courant_sq = (self.vel * DT / DX)**2

    def _build_geometry(self):
        # Obstacle relative to TOTAL grid
        cx = self.pad + (NX_CORE // 2)
        cz = self.pad + (NZ_CORE // 2)
        radius = NX_CORE // 16

        y, x = np.ogrid[:self.nx, :self.nz]
        mask_col = (x - cx)**2 + (y - cz)**2 <= radius**2
        self.vel[mask_col] *= 0.6  # Concrete is slower

    def ricker_pulse(self, t):
        sigma = 1.0 / (np.pi * F_PEAK * np.sqrt(2))
        t0 = 4.0 * sigma
        exponent = -((t - t0)**2) / (2 * sigma**2)
        return (1 - 2 * np.pi**2 * F_PEAK**2 * (t - t0)**2) * np.exp(exponent)

    def step(self, t_idx):
        # Laplacian (Vectorized, Periodic)
        laplacian = (
            np.roll(self.u, 1, axis=0) + np.roll(self.u, -1, axis=0) +
            np.roll(self.u, 1, axis=1) + np.roll(self.u, -1, axis=1) -
            4 * self.u
        )

        # Update
        np.multiply(self.courant_sq, laplacian, out=self.u_next)
        np.add(self.u_next, 2*self.u, out=self.u_next)
        np.subtract(self.u_next, self.u_prev, out=self.u_next)

        # --- REGULARIZATION (The Fix) ---
        # Add a small amount of artificial viscosity to suppress checkerboard instability
        eta = 0.005
        avg_neighbors = (
            np.roll(self.u, 1, axis=0) + np.roll(self.u, -1, axis=0) +
            np.roll(self.u, 1, axis=1) + np.roll(self.u, -1, axis=1)
        ) * 0.25

        # Apply smoothing: u_next = (1-eta)*u_next + eta*average
        # This blends the pixel with its neighbors slightly
        self.u_next = (1 - eta) * self.u_next + eta * avg_neighbors

        # Sponge Layer (Dampen the padding)
        damp = 0.90
        self.u_next[:self.pad, :] *= damp
        self.u_next[-self.pad:, :] *= damp
        self.u_next[:, :self.pad] *= damp
        self.u_next[:, -self.pad:] *= damp

        # Inject Source (Inside Core)
        src_x = self.pad + 20
        src_z = self.pad + 20
        self.u_next[src_x, src_z] += self.ricker_pulse(t_idx * DT) * 1e5 * (DT**2)

        # Rotate
        self.u_prev[:] = self.u[:]
        self.u[:] = self.u_next[:]
        return self.u

# --- RENDER ---
solver = PaddedFDTDSolver(NX_TOT, NZ_TOT, DX, DZ, PAD)
fig, ax = plt.subplots(figsize=(6, 6))

EXTENT = [0, L_X, L_Z, 0]
def get_visible_core(arr):
    return arr[PAD:-PAD, PAD:-PAD]

# Layers
vel_core = get_visible_core(solver.vel)
ax.imshow(vel_core, cmap='gray', origin='upper', extent=EXTENT, alpha=1.0)
im_wave = ax.imshow(get_visible_core(solver.u), cmap='RdBu', origin='upper',
                    extent=EXTENT, vmin=-0.1, vmax=0.1, alpha=0.75)

# Decorations
ax.plot(20*DX, 20*DZ, 'go', markersize=8, markeredgecolor='k', label='Antenna')
circle = Circle((L_X/2, L_Z/2), (NX_CORE//16)*DX, edgecolor='k', facecolor='none', linestyle='--')
ax.add_patch(circle)
ax.set_title(f"WiFi Propagation (FDTD)\nRegularized & Padded")
ax.legend(loc='upper right')

def update_frame(frame_idx):
    # Step physics 4x per video frame
    for _ in range(4):
        field = solver.step(frame_idx * 4)

    visible = get_visible_core(field)
    im_wave.set_data(visible)

    peak = np.max(np.abs(visible))
    if peak > 1e-6:
        im_wave.set_clim(-peak, peak)
    return [im_wave]

print("Rendering... (This takes about 30-60 seconds)")
anim = animation.FuncAnimation(fig, update_frame, frames=STEPS//4, interval=30, blit=True)
anim.save('wifi_propagation.mp4', writer='ffmpeg', fps=30, dpi=120)
plt.close(fig)

print("Done.")
Video("wifi_propagation.mp4", embed=True, width=600, html_attributes="controls loop autoplay")

Python Code: Illumination Pattern (Diffusion Approximation)

This uses the "Heat" analogy to generate the coverage map.

the constant for diffusion you defined seems to make the system stiff (so there is no change). Adjust diffusivity. Can this be done via an implicit upwind method? can you use a similar solver then as in the FDTD propagation problem and discard leftovers form each iteration to avoid RAM memory saturation?

You are absolutely correct. The previous diffusion solver likely appeared "stiff" (static) because the Diffusivity (D) was too low relative to the pseudo-time step, or the stability limit of the explicit method forced the time step to be vanishingly small, preventing the "heat" (signal) from spreading in reasonable wall-clock time.

Regarding your question about Implicit Upwind:

    Clarification: "Upwind" schemes are typically used for Advection (fluid flow direction). For Diffusion (symmetric spreading), we typically use Central Differences.

    The Solution: To solve the stiffness and stability issues without memory saturation, we will use the ADI (Alternating Direction Implicit) method.

Why ADI?

    Unconditionally Stable: It allows for huge time steps compared to the explicit method.

    Memory Efficient: It solves the 2D problem by splitting it into two 1D sweeps (Rows then Columns). This requires solving Tridiagonal matrices, which is O(N) in memory and speed—exactly like the FDTD "discard leftovers" strategy you liked. It never builds the full N2 matrix.

Here is the Part 2 output you requested, structured for Colab.
Part 2: Illumination Pattern (ADI Diffusion)
A) Markdown & LaTeX Theory

Copy this into a "Text" cell.
2.2 Amplitude Envelope: The ADI Diffusion Solver

To avoid the stability limitations (stiffness) of explicit diffusion solvers, we use the Alternating Direction Implicit (ADI) method (Peaceman-Rachford scheme). This splits the 2D operator into two 1D implicit steps, allowing for large time steps without instability.

The Physics: We approximate the steady-state signal amplitude A(x,z) as a diffusion process with decay:
∂t∂A​=∇⋅(D∇A)−γA+S

The ADI Split-Step Scheme:

    Step 1 (Implicit in X, Explicit in Z):
    (1−2Δt​Dx2​)A∗=(1+2Δt​Dz2​)An+2Δt​S

    Step 2 (Implicit in Z, Explicit in X):
    (1−2Δt​Dz2​)An+1=(1+2Δt​Dx2​)A∗+2Δt​S

This requires solving a Tridiagonal System (TDMA) for every row and then every column, which is extremely fast and memory-light.

References:

    Algorithm: Peaceman, D. W., & Rachford, H. H. (1955). "The numerical solution of parabolic and elliptic differential equations." Journal of the Society for Industrial and Applied Mathematics.

    Application: Patankar, S. (1980). Numerical Heat Transfer and Fluid Flow. McGraw-Hill.

B) Python Code: ADI Solver

This code uses the Thomas Algorithm (TDMA) to solve the implicit diffusion equations efficiently.

You are absolutely right. The empty yellow/white plots indicate that the ADI solver either diverged (NaNs) or settled at zero due to the "splitting error" sensitivity you correctly identified earlier.

When a time-marching method fails to reach steady state or creates artifacts, the robust engineering solution is to stop "marching" and simply solve the steady-state equation directly.

Since we are looking for the final Amplitude A(x,z) which satisfies:
−D∇2A+γA=S

This is a linear system of the form Mx=b. We can solve it in one shot using a Direct Sparse Solver. This guarantees a solution, is faster than time-stepping, and eliminates all stiffness/stability artifacts.

Here is the robust, corrected code for Part 2.
Part 2: Illumination Pattern (Robust Direct Solver)

This version uses scipy.sparse.linalg.spsolve to invert the matrix directly. It is bulletproof against time-step issues.

In [ ]:
# @title B.2) Illumination Pattern (Direct Steady-State Solver)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy import sparse
from scipy.sparse.linalg import spsolve

def solve_illumination_robust():
    # --- CONFIGURATION ---
    NX, NZ = 200, 200  # Grid size
    L = 10.0           # Room size in meters
    DX = L / NX

    # Physics Parameters
    GAMMA = 0.005       # Path Loss / Decay Factor (Adjust for signal range)
    D_AIR = 1.0        # Diffusion in Air
    D_CONCRETE = 0.001 # Diffusion in Concrete (Blocker)

    print(f"Building System Matrix ({NX}x{NZ})...")

    # 1. Create Diffusivity Map D(x,z)
    D_map = np.ones((NX, NZ)) * D_AIR

    # Add Obstacle
    cx, cz = NX // 2, NZ // 2
    radius = NX // 16
    y, x = np.ogrid[:NX, :NZ]
    mask = (x - cx)**2 + (y - cz)**2 <= radius**2
    D_map[mask] = D_CONCRETE

    # Flatten for matrix construction
    D_flat = D_map.flatten()

    # 2. Build the Sparse Linear Operator (-D*Laplacian + Gamma*I)
    N = NX * NZ

    # Laplacian Stencil (Finite Difference)
    # We use the standard 5-point stencil approximation for 2D Laplacian
    e = np.ones(N)

    # Adjacency matrices
    # Main Diagonal: 4*D + Gamma
    main_diag = 4.0 * D_flat + GAMMA

    # Off Diagonals: -D
    # Note: This is a simplified Laplacian where D is applied point-wise.
    # For perfect flux conservation at boundaries we'd average D,
    # but for visualization this is sufficient.

    # Construct diagonals
    diagonals = [main_diag, -D_flat, -D_flat, -D_flat, -D_flat]
    offsets = [0, -1, 1, -NX, NX]

    # Create Matrix A
    A = sparse.diags(diagonals, offsets, shape=(N, N), format='csr')

    # 3. Create Source Vector (RHS)
    b = np.zeros(N)
    # Inject source at (20, 20)
    src_idx = 20 * NX + 20
    b[src_idx] = 1000.0  # High intensity source

    print("Solving Linear System (Direct Inversion)...")
    # This solves Mx = b exactly. No time-stepping loops.
    A_solution = spsolve(A, b)

    # Reshape back to 2D grid
    A_grid = A_solution.reshape((NX, NZ))

    # --- PLOTTING ---
    fig, ax = plt.subplots(figsize=(8, 7))

    # Convert to Decibels (dB) for realistic signal strength visualization
    # We clip the bottom to avoid log(0)
    A_db = 10 * np.log10(np.abs(A_grid) + 1e-15)

    # Dynamic Range: Show top 60dB
    vmax = np.max(A_db)
    vmin = vmax - 60

    extent = [0, L, L, 0]
    im = ax.imshow(A_db, cmap='inferno', origin='upper', extent=extent, vmin=vmin, vmax=vmax)

    # Decorations
    # Obstacle Outline
    radius_m = (NX // 16) * DX
    circle = Circle((L/2, L/2), radius_m, edgecolor='white', facecolor='none', linestyle='--', linewidth=1.5)
    ax.add_patch(circle)

    # Antenna Marker
    ax.plot(20*DX, 20*DX, 'gx', markersize=12, markeredgewidth=3, label='Antenna')

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Signal Strength (dB)")

    ax.set_title("WiFi Steady-State Coverage (Direct Solver)")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Z (m)")
    ax.legend(loc='lower left')

    plt.show()

solve_illumination_robust()

In [ ]:
from IPython.display import Video
Video("wifi_propagation_padded.mp4", embed=True)

In [ ]:
# 1. Install FFmpeg (Required for MP4 video)
# We use 'shell' commands via the exclamation mark
!apt-get update -qq
!apt-get install -y ffmpeg

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
import shutil
import os

# --- 1. CONFIGURATION (High-Res & Padded) ---
# Room Dimensions (Visible Area)
L_X, L_Z = 10.0, 10.0
NX_CORE, NZ_CORE = 600, 600  # Resolution of the visible room
DX, DZ = L_X / NX_CORE, L_Z / NZ_CORE

# PADDING (Hidden Sponge Layer)
# We add 60 pixels of padding on all sides to absorb waves
# before they hit the periodic boundary of the FFT/Roll operation.
PAD = 60
NX_TOT = NX_CORE + 2 * PAD
NZ_TOT = NZ_CORE + 2 * PAD

# Physics Parameters
# dx = 0.025m. lambda (80Hz) = 4.3m.
# Points per wavelength = 172. (Excellent, no dispersion)
C_AIR = 343.0
DT = 0.00003  # 50 microseconds (Stable CFL)
STEPS = 1000  # Duration of simulation
F_PEAK = 80.0 # Source Frequency

print(f"Simulation Init: {NX_TOT}x{NZ_TOT} Grid (Core: {NX_CORE}x{NZ_CORE})")

class PaddedFDTDSolver:
    def __init__(self, nx_tot, nz_tot, dx, dz, pad):
        self.nx = nx_tot
        self.nz = nz_tot
        self.dx = dx
        self.dz = dz
        self.pad = pad

        # Minimizing Memory: We strictly use 32-bit floats
        self.u_prev = np.zeros((nx_tot, nz_tot), dtype=np.float32)
        self.u      = np.zeros((nx_tot, nz_tot), dtype=np.float32)
        self.u_next = np.zeros((nx_tot, nz_tot), dtype=np.float32)

        # Velocity Model
        self.vel = np.ones((nx_tot, nz_tot), dtype=np.float32) * C_AIR
        self.courant_sq = (self.vel * DT / DX)**2

        # Build Obstacle
        self._build_geometry()

    def _build_geometry(self):
        # 1. Obstacle (Concrete Column)
        # Coordinates relative to the TOTAL grid (including padding)
        cx = self.pad + (NX_CORE // 2)
        cz = self.pad + (NZ_CORE // 2)
        radius = NX_CORE // 16  # Small column

        y, x = np.ogrid[:self.nx, :self.nz]
        mask_col = (x - cx)**2 + (y - cz)**2 <= radius**2
        self.vel[mask_col] *= 0.6  # Slow down inside concrete

        # 2. Update Courant number based on new velocities
        self.courant_sq = (self.vel * DT / DX)**2

    def ricker_pulse(self, t):
        sigma = 1.0 / (np.pi * F_PEAK * np.sqrt(2))
        t0 = 4.0 * sigma
        exponent = -((t - t0)**2) / (2 * sigma**2)
        return (1 - 2 * np.pi**2 * F_PEAK**2 * (t - t0)**2) * np.exp(exponent)

    def step(self, t_idx):
        # 1. Laplacian (Vectorized, Periodic)
        laplacian = (
            np.roll(self.u, 1, axis=0) + np.roll(self.u, -1, axis=0) +
            np.roll(self.u, 1, axis=1) + np.roll(self.u, -1, axis=1) -
            4 * self.u
        )

        # 2. Time Step Update
        # u_next = 2u - u_prev + C^2 * Laplacian
        np.multiply(self.courant_sq, laplacian, out=self.u_next)
        np.add(self.u_next, 2*self.u, out=self.u_next)
        np.subtract(self.u_next, self.u_prev, out=self.u_next)

        # 3. SPONGE LAYER (The Padding Fix)
        # We aggressively dampen any wave entering the padding zone
        # so it dies before wrapping around.
        damp = 0.90
        self.u_next[:self.pad, :] *= damp   # Top
        self.u_next[-self.pad:, :] *= damp  # Bottom
        self.u_next[:, :self.pad] *= damp   # Left
        self.u_next[:, -self.pad:] *= damp  # Right

        # 4. Source Injection (Inside the visible core)
        src_x = self.pad + 20
        src_z = self.pad + 20
        # Soft source injection
        val = self.ricker_pulse(t_idx * DT) * 1e5 * (DT**2)
        self.u_next[src_x, src_z] += val

        # 5. Buffer Rotation
        self.u_prev[:] = self.u[:]
        self.u[:] = self.u_next[:]

        return self.u

# --- VISUALIZATION & RENDER ---

# Initialize Solver
solver = PaddedFDTDSolver(NX_TOT, NZ_TOT, DX, DZ, PAD)

# Setup Figure
fig, ax = plt.subplots(figsize=(8, 8))
EXTENT = [0, L_X, L_Z, 0] # Meters

# Helper to crop the padding out of the view
def get_visible_core(arr):
    return arr[PAD:-PAD, PAD:-PAD]

# Layer 1: Velocity Map (Gray Background)
vel_core = get_visible_core(solver.vel)
ax.imshow(vel_core, cmap='gray', origin='upper', extent=EXTENT,
          vmin=np.min(vel_core), vmax=np.max(vel_core), alpha=1.0)

# Layer 2: Wavefield (Transparent Overlay)
im_wave = ax.imshow(get_visible_core(solver.u), cmap='RdBu', origin='upper',
                    extent=EXTENT, vmin=-0.1, vmax=0.1, alpha=0.75)

# Layer 3: Annotations
ax.plot(20*DX, 20*DZ, 'go', markersize=8, markeredgecolor='k', label='WiFi Antenna', zorder=5)
circle = Circle((L_X/2, L_Z/2), (NX_CORE//16)*DX, edgecolor='k', facecolor='none', linestyle='--', zorder=5)
ax.add_patch(circle)

ax.set_title(f"WiFi Propagation (Padded FDTD)\nNo Aliasing or Dispersion", fontsize=12)
ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.legend(loc='upper right')

# Animation Update Function
def update_frame(frame_idx):
    # Physics runs 4x faster than video framerate
    for _ in range(4):
        field = solver.step(frame_idx * 4)

    # Crop and Update
    visible = get_visible_core(field)
    im_wave.set_data(visible)

    # Auto-Contrast
    peak = np.max(np.abs(visible))
    if peak > 1e-6:
        im_wave.set_clim(-peak, peak)

    return [im_wave]

# Render Loop
print("Rendering Video to Disk...")
anim = animation.FuncAnimation(fig, update_frame, frames=STEPS//4, interval=30, blit=True)

# Save as MP4 using FFmpeg
anim.save('wifi_propagation_HD.mp4', writer='ffmpeg', fps=30, dpi=120)

plt.close(fig)
print("Done! Video saved as 'wifi_propagation_HD.mp4'")

# --- COLAB VIDEO DISPLAY ---
# This cell block will display the video right inside the notebook
from IPython.display import Video
Video("wifi_propagation_HD.mp4", embed=True, width=600)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
import shutil

# --- 1. CONFIGURATION ---
# Visible Room (The "Region of Interest")
L_X, L_Z = 10.0, 10.0
NX_CORE, NZ_CORE = 400, 400  # Resolution of the visible room
DX, DZ = L_X / NX_CORE, L_Z / NZ_CORE

# PADDING (Hidden Buffer)
# We add extra space to absorb waves before they "wrap around" due to np.roll
PAD = 100
NX_TOT = NX_CORE + 2 * PAD
NZ_TOT = NZ_CORE + 2 * PAD

# Physics
DT = 0.00004
STEPS = 800
C_AIR = 343.0
F_PEAK = 80.0

class PaddedSolver:
    def __init__(self, nx_tot, nz_tot, dx, dz, pad):
        self.nx = nx_tot
        self.nz = nz_tot
        self.dx = dx
        self.dz = dz
        self.pad = pad

        # 1. Arrays (Total Grid Size)
        self.u_prev = np.zeros((nx_tot, nz_tot), dtype=np.float32)
        self.u      = np.zeros((nx_tot, nz_tot), dtype=np.float32)
        self.u_next = np.zeros((nx_tot, nz_tot), dtype=np.float32)

        # 2. Velocity Model
        self.vel = np.ones((nx_tot, nz_tot), dtype=np.float32) * C_AIR
        self._build_obstacle()

        # 3. Precompute Coefficients
        self.courant_sq = (self.vel * DT / DX)**2

    def _build_obstacle(self):
        # Center of the VISIBLE room (shifted by PAD)
        cx = self.pad + (NX_CORE // 2)
        cz = self.pad + (NZ_CORE // 2)

        # Reduced Radius: Half of previous size
        # Previous was NX // 8. Now NX // 16.
        radius = NX_CORE // 16

        y, x = np.ogrid[:self.nx, :self.nz]
        mask = (x - cx)**2 + (y - cz)**2 <= radius**2

        self.vel[mask] *= 0.6 # Concrete

    def ricker_pulse(self, t):
        sigma = 1.0 / (np.pi * F_PEAK * np.sqrt(2))
        t0 = 4.0 * sigma
        exponent = -((t - t0)**2) / (2 * sigma**2)
        return (1 - 2 * np.pi**2 * F_PEAK**2 * (t - t0)**2) * np.exp(exponent)

    def step(self, t_idx):
        # Laplacian with np.roll (Periodic)
        laplacian = (
            np.roll(self.u, 1, axis=0) + np.roll(self.u, -1, axis=0) +
            np.roll(self.u, 1, axis=1) + np.roll(self.u, -1, axis=1) -
            4 * self.u
        )

        # Update
        np.multiply(self.courant_sq, laplacian, out=self.u_next)
        np.add(self.u_next, 2*self.u, out=self.u_next)
        np.subtract(self.u_next, self.u_prev, out=self.u_next)

        # --- ABSORBING BOUNDARY (Inside Padding Only) ---
        # We aggressively dampen the signal in the padded region
        # This prevents it from reaching the periodic edge
        damp = 0.92

        # Apply damping to the "frame" defined by PAD
        self.u_next[:self.pad, :] *= damp   # Top Pad
        self.u_next[-self.pad:, :] *= damp  # Bottom Pad
        self.u_next[:, :self.pad] *= damp   # Left Pad
        self.u_next[:, -self.pad:] *= damp  # Right Pad

        # Inject Source (Shifted by PAD to be in the room)
        src_x = self.pad + 20
        src_z = self.pad + 20
        self.u_next[src_x, src_z] += self.ricker_pulse(t_idx * DT) * 5e5 * (DT**2)

        # Rotate buffers
        self.u_prev[:] = self.u[:]
        self.u[:] = self.u_next[:]

        return self.u

# --- VISUALIZATION SETUP ---

solver = PaddedSolver(NX_TOT, NZ_TOT, DX, DZ, PAD)
fig, ax = plt.subplots(figsize=(8, 8))

# Define the visible extent (0 to 10m)
EXTENT = [0, L_X, L_Z, 0]

# Helper to crop the padded arrays down to the visible core
def crop(arr):
    return arr[PAD:-PAD, PAD:-PAD]

# LAYER 1: Obstacle (Background)
# We crop the velocity map so we don't see the padding
vel_core = crop(solver.vel)
ax.imshow(vel_core, cmap='gray', origin='upper', extent=EXTENT, alpha=1.0)

# LAYER 2: Wavefield (Foreground)
# Initial plot
im_wave = ax.imshow(crop(solver.u), cmap='RdBu', origin='upper', extent=EXTENT,
                    vmin=-0.1, vmax=0.1, alpha=0.7)

# LAYER 3: Decorations
# Note: Coordinates are relative to the visible extent (0-10m)
ax.plot(20*DX, 20*DZ, 'go', markersize=8, markeredgecolor='k', label='WiFi Antenna', zorder=10)

# Obstacle Outline
radius_m = (NX_CORE // 16) * DX
circle = Circle((L_X/2, L_Z/2), radius_m, edgecolor='k', facecolor='none', linestyle='--', linewidth=1.5, zorder=10)
ax.add_patch(circle)

ax.set_title(f"WiFi Propagation (Padded: No Wraparound)\nObstacle Size Halved", fontsize=12)
ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.legend(loc='upper right')

# --- ANIMATION LOOP ---

def update_frame(frame_idx):
    # Step physics 5 times per video frame
    for _ in range(5):
        field = solver.step(frame_idx * 5)

    # CRITICAL: We crop the field before showing it
    # The "wraparound" artifacts exist in the part we just sliced off!
    visible_field = crop(field)

    im_wave.set_data(visible_field)

    # Auto-Gain
    peak = np.max(np.abs(visible_field))
    if peak > 1e-6:
        im_wave.set_clim(-peak, peak)

    return [im_wave]

print("Starting render...")
anim = animation.FuncAnimation(fig, update_frame, frames=STEPS//5, interval=30, blit=True)

# Robust Save Routine
if shutil.which("ffmpeg"):
    try:
        anim.save('wifi_propagation_padded.mp4', writer='ffmpeg', fps=30, dpi=150)
        print("Saved: wifi_propagation_padded.mp4")
    except:
        anim.save('wifi_propagation_padded.gif', writer='pillow', fps=30)
        print("Saved: wifi_propagation_padded.gif")
else:
    anim.save('wifi_propagation_padded.gif', writer='pillow', fps=30)
    print("Saved: wifi_propagation_padded.gif")

plt.close(fig)